In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab Notebooks/RAG with Excel and PDF Data with Langchain and OpenAI

/content/drive/MyDrive/Colab Notebooks/RAG with Excel and PDF Data with Langchain and OpenAI


In [3]:
from google.colab import userdata
api_key = userdata.get('Groq')

# **EXCEL**


In [4]:
!pip install langchain-community unstructured langchain-groq faiss-cpu sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━

In [5]:
#Import libraries
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores.faiss import FAISS
from IPython.display import display, Markdown

In [20]:
loader = UnstructuredExcelLoader("Sample of Employee Database.xlsx")
data = loader.load()
data[:5]

[Document(metadata={'source': 'Sample of Employee Database.xlsx'}, page_content='Term Status Employee Name Department Function Title Direct Manager Joining Date End of Probation Business Email Phone Number Model Office Area National ID Gender DOB Age Bank Account Number Bank name Car Major Level of Exp Emergency Num Relation Social Insurance Medical Insurance Resignartion date Probation apprasil Documents Fulltime Hired Ahmed Saeed Mohamed Ibrahim Administation Office Support Mohamed Magdy 2019-02-01 00:00:00 2019-05-01 00:00:00 1116490906 Onsite Cairo 29410102104199 Male 1994-10-10 00:00:00 30 Cash No 1149611945 Wife Yes Yes Fulltime Hired Ramadan Abdel Bary Maarouf Abdelbary Administation Office Support Mohamed Magdy 2019-06-01 00:00:00 2019-09-01 00:00:00 1127787500 Onsite Cairo 29310282100213 Male 1993-10-28 00:00:00 31 Cash No 1146442479 Brother Yes Yes Fulltime Hired Mohamed Magdy Badawy Ahmed Administation Facility Supervisor Nourhan Mousa 2019-08-01 00:00:00 2019-11-01 00:00:00

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
chunks = text_splitter.split_documents(data)
chunks[0]

Document(metadata={'source': 'Sample of Employee Database.xlsx'}, page_content='Term Status Employee Name Department Function Title Direct Manager Joining Date End of Probation Business Email Phone Number Model Office Area National ID Gender DOB Age Bank Account Number Bank name Car Major Level of Exp Emergency Num Relation Social Insurance Medical Insurance Resignartion date Probation apprasil Documents Fulltime Hired Ahmed Saeed Mohamed Ibrahim Administation Office Support Mohamed Magdy 2019-02-01 00:00:00 2019-05-01 00:00:00 1116490906 Onsite Cairo 29410102104199 Male 1994-10-10 00:00:00 30 Cash No 1149611945 Wife Yes Yes Fulltime Hired Ramadan Abdel Bary Maarouf Abdelbary Administation Office Support Mohamed Magdy 2019-06-01 00:00:00 2019-09-01 00:00:00 1127787500 Onsite Cairo 29310282100213 Male 1993-10-28 00:00:00 31 Cash No 1146442479 Brother Yes Yes Fulltime Hired Mohamed Magdy Badawy Ahmed Administation Facility Supervisor Nourhan Mousa 2019-08-01 00:00:00 2019-11-01 00:00:00 

In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db_faiss = FAISS.from_documents(chunks, embeddings)

/tmp/ipython-input-8-1803624520.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models o

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
query = "Retrieve the worst reviews"
output_retrieval = db_faiss.similarity_search(query, k=3) # Reduced k from 5 to 2
output_retrieval_merged = "\n\n".join([doc.page_content for doc in output_retrieval])
display(Markdown(output_retrieval_merged))

Hired Refaat Ashraf Refaat Batal Technology Data Senior L1 Data Analyst Mohamed Barkat 2025-05-07 00:00:00 2025-08-07 00:00:00 refaat.batal@paysky.io 1553071348 Onsite Cairo 29411260101331 Male 1994-11-26 00:00:00 30 100058209761 CIB Yes 1279016193 Father Yes Yes Fulltime Hired Mohamed elsayed Elewa Salem Shalaby Marketing Customer Success Customer Success Specialist - PAK Essam Oraby 2023-06-11 00:00:00 2023-09-11 00:00:00 mohamed.elewa@paysky.io 1061421111 Onsite Cairo 28511201401194 Male 1985-11-20 00:00:00 39 100049201828 CIB 1027002036 Wife Yes Yes Fulltime Hired Omar Ahmed Tarek Ibrahim Mohamed Marketing BTL Supervisor Hisham Moharam 2023-07-03 00:00:00 2023-10-03 00:00:00 omar.tarek@paysky.io 1140486447 Onsite Cairo 29802140103614 Male 1998-02-14 00:00:00 27 100060193837 CIB Yes 1223484792 Father Yes Yes Fulltime Hired Salwa Amr Beshir Mousa Ahmed Marketing Customer Success Customer Success Specialist Khloud Khaled 2023-08-06 00:00:00 2023-11-06 00:00:00 salwa.amr@yalla.online 1152475006 Hybrid Cairo 29601160102929 Female 1996-01-16 00:00:00 29 100051385057 CIB No 1148088488 Husband No Yes Fulltime Hired Mohamed Hesham Mahdy Nour Marketing Customer Success Customer Success Specialist Khloud Khaled 2023-09-17 00:00:00 2023-12-17 00:00:00 mohamed.hesham@paysky.io 1094801964 Hybrid Cairo 29812031700454 Male 1998-12-03 00:00:00 26 100060193198 CIB No 1021195792 Father Yes Yes Fulltime Hired Medhat Adel Hassan Aly Marketing Customer Success Customer Success Supervisor Khloud Khaled 2023-09-24 00:00:00 2023-12-24 00:00:00 medhat.adel@paysky.io 1557606000 Hybrid Cairo 28403262100092 Male 1984-03-26 00:00:00 41 100049201545 CIB No 1063017821 Wife Yes Yes Fulltime Hired Moustafa Hussien Aly Hussien Abo Gabal Marketing Data Analyst Hassan Dorgham 2023-10-01 00:00:00 2024-01-01 00:00:00 Mostafa.hussein@paysky.io 1020277199 Onsite Cairo 29304090103219 Male 1993-04-09 00:00:00 32 100030168539 CIB Yes 1003628251 Sister Yes Yes Parttime / Freelance Resigned Mohamed Sabry Mohamed Eissa Marketing Senior Graphic Designer Hisham Moharam 2023-12-31 00:00:00 2024-03-31 00:00:00 mohamed.sabri@paysky.io 1149732006 Onsite Cairo 29207190100556 Male 1992-07-19 00:00:00 32 100061371316 CIB No 1065845220 Sister Yes Yes 2025-05-15 00:00:00 Fulltime Hired Amira Essam Elgohary Marketing Customer Success Customer Success Specialist Khloud Khaled 2024-02-15 00:00:00 2024-05-15 00:00:00 amira.essam@yalla.online 1064478223 Hybrid Cairo 29609198800825 Female 1996-09-19 00:00:00 28 100039325556 CIB No 1098979415 Husband No Yes Fulltime Hired AbdelHady Adel Helal Sayed Ahmed Marketing Customer Success Senior Customer Success Specialist Khloud Khaled 2024-08-01 00:00:00 2024-11-01 00:00:00 abdelhady.helal@paysky.io 1100869116 Hybrid Cairo 29606120105277 Male 1996-06-12 00:00:00 28 100053372745 CIB 1023774808 Wife Yes Yes Fulltime Hired Omar Hassan Abdelkhalek Hassan Marketing Customer Success Senior Customer Success Specialist Khloud Khaled 2024-09-01 00:00:00 2024-12-01 00:00:00 omar.hassan@paysky.io 1158639668 Hybrid Cairo 29610030101776 Male 1996-10-03 00:00:00 28 100055681567 CIB 1146016632 Father Yes Yes Fulltime Hired Aliaa Mostafa Mohamed Abdelmoniem Badway Marketing Social Media Specialist Hisham Moharam 2024-12-01 00:00:00 2025-03-01 00:00:00 aliaa.mostafa@paysky.io 1000700826 Onsite Cairo 29809010100707 Female 1998-09-01 00:00:00 26 100064926898 CIB 1000013915 sister Yes Yes Birth C, Graduation C, Criminal C, Photos, ID Fulltime Hired Ameer Hamza Malik Marketing Customer Success Customer Success Specialist - PAK Essam Oraby 2024-12-02 00:00:00 2025-03-02 00:00:00 ameer.hamza@paysky.io 923320564533 Remote Pakistan Islamabad Male 125 Cash No No Fulltime Hired Hany Nabil AbdelHalim Mohamed Marketplace Merchant Acquisition Supervisor 2021-10-13 00:00:00 2022-01-13 00:00:00 hany.nabil@paysky.io 1099258907 Onsite Cairo 28307241402634 Male 1983-07-24 00:00:00 41 100050424647 CIB Yes 1068051024 Wife Yes Yes Fulltime Resigned Mahmoud Hamdy AbdelGabar Abdelaal Marketplace Marketplace Manager Mahmoud Taha 2022-07-24 00:00:00 2022-10-24 00:00:00 Mahmoud.Hamdy@paysky.io 1111119890 Onsite Cairo 28603160103614 Male 1986-03-16 00:00:00 39 100054775681 CIB Yes 1097833631 Wife Yes Yes 2025-05-20 00:00:00 Fulltime Hired Duaa Elsayed Hassan Abdelwarith Marketplace Senior L1 Content Writer Moustafa Ashraf 2023-02-12 00:00:00 2023-05-12 00:00:00 duaa.abdelwarith@paysky.io 1155674069 Onsite Cairo 29810188800387 Female 1998-10-18 00:00:00 26 100058209998 CIB Yes 1155674320 Brother Yes Yes Fulltime Hired Moustafa Ashraf Farouk Hassan Elzohery Marketplace Category lead Mahmoud Taha 2023-05-01 00:00:00 2023-08-01 00:00:00 mostafa.ashraf@paysky.io 1111124749 Onsite Cairo 29201040101297 Male 1992-01-04 00:00:00 33 100058209222 CIB 1141006679 Wife Yes Yes Fulltime Hired Ramy Mohamed Mahmoud Aly Marketplace Operations Manager Mahmoud Taha 2023-08-06 00:00:00 2023-11-06 00:00:00 ramy.mohamed@paysky.io 1000100865 Onsite Cairo 28608240102271 Male 1986-08-24 00:00:00 38 100044060897 CIB Yes 1281188996 Wife No Yes Fulltime Hired Yara Aahed Rohy Ahmed Elsharfa Marketplace Content Creator Moustafa Ashraf 2023-09-03 00:00:00 2023-12-03 00:00:00 yara.rawhy@paysky.io 1015729234 Onsite Cairo 30108010112249 Female 2001-08-01 00:00:00 23 100060193578 CIB No 1270415641 Mother No Yes Parttime / Freelance Hired Salma Moustafa Badreldin Moustafa Besher Marketplace Account Manager 2023-11-01 00:00:00 2024-02-01 00:00:00 salma.moustafa@paysky.io 1287374301 Onsite Cairo 30007010104288 Female 2000-07-01 00:00:00 24 100062026752 CIB No 1097200189 Sister No Yes 2025-06-30 00:00:00 Fulltime Resigned Mennaallah Farid Mohamed Mousa Marketplace Content Creator Duaa Abdelwarith 2023-11-01 00:00:00 2024-02-01 00:00:00 menna.moussa@paysky.io 1144097111 Onsite Cairo 30105280100804 Female 2001-05-28 00:00:00 24 100062023435 CIB Yes 1144095777 Father Yes Yes 2025-04-30 00:00:00 Internship Hired Osama Yasser Mohamed Fahmy Mohamed Marketing Marketing Intern Hisham Moharam 2025-05-11 00:00:00 2025-08-11 00:00:00 osamaayasser17@gmail.com 1028453305 Onsite Cairo 30502100101572 Male 2005-02-10 00:00:00 20 855100897339101000 NBE 1229791999 Father No No

Term Status Employee Name Department Function Title Direct Manager Joining Date End of Probation Business Email Phone Number Model Office Area National ID Gender DOB Age Bank Account Number Bank name Car Major Level of Exp Emergency Num Relation Social Insurance Medical Insurance Resignartion date Probation apprasil Documents Fulltime Hired Ahmed Saeed Mohamed Ibrahim Administation Office Support Mohamed Magdy 2019-02-01 00:00:00 2019-05-01 00:00:00 1116490906 Onsite Cairo 29410102104199 Male 1994-10-10 00:00:00 30 Cash No 1149611945 Wife Yes Yes Fulltime Hired Ramadan Abdel Bary Maarouf Abdelbary Administation Office Support Mohamed Magdy 2019-06-01 00:00:00 2019-09-01 00:00:00 1127787500 Onsite Cairo 29310282100213 Male 1993-10-28 00:00:00 31 Cash No 1146442479 Brother Yes Yes Fulltime Hired Mohamed Magdy Badawy Ahmed Administation Facility Supervisor Nourhan Mousa 2019-08-01 00:00:00 2019-11-01 00:00:00 mohamed.magdy@paysky.io 1020090082 / 1008021119 Onsite Cairo 28502182100693 Male 1985-02-18 00:00:00 40 100027973668 CIB Yes 1096128789 Wife Yes Yes Fulltime Hired Ramadan Fathy Omar Mohamed Administation Office Support Mohamed Magdy 2019-10-20 00:00:00 2020-01-20 00:00:00 1146902513 Onsite Cairo 29304052100371 Male 1993-04-05 00:00:00 32 Cash No 1127506077 Wife Yes Yes Fulltime Hired Mohamed Abdel Nour Mohamed Administation Admin Supervisor Ahmed Salem 2020-11-01 00:00:00 2021-02-01 00:00:00 1065547789 Onsite Cairo 27811020101334 Male 1978-11-02 00:00:00 46 100045931317 CIB No 1150792628 Wife Yes Yes Fulltime Hired Mohamed Abdelhalim Maarouf Abdelbary Administation Office Support Mohamed Magdy 2023-08-20 00:00:00 2023-11-20 00:00:00 1110250534 Onsite Cairo 29512132100152 Male 1995-12-13 00:00:00 29 Cash No 1128604142 Wife No Yes Fulltime Hired Mohamed Essam Adly ElBatawy Business Development Ecommerce Acquisition Lead Karim Khattab 2021-03-01 00:00:00 2021-06-01 00:00:00 mohamed.elbatawy@paysky.io 1017518030 Onsite Cairo 29608181700855 Male 1996-08-18 00:00:00 28 100039870341 CIB Yes 1007191759 Mother Yes Yes Fulltime Hired Karim Salah Abdelwahab Ahmed Khattab Business Development Chief Business Development Officer - CBDO Waleed Sadek 2023-08-27 00:00:00 2023-11-27 00:00:00 kareem.khattab@paysky.io 1026206868 Hybrid Cairo Madinty 28510218800894 Male 1985-10-21 00:00:00 39 100038626617 CIB Yes 1000372289 Wife Yes Yes Fulltime Hired Ibrahim Azmy Mohamed Mohamed Finance & Accounting Chief Financial Officer - CFO Waleed Sadek 2013-04-15 00:00:00 2013-07-15 00:00:00 ibrahim.azmy@paysky.io 1029999845 Onsite Cairo 28108191402015 Male 1981-08-19 00:00:00 43 100033799459 CIB Yes 1001836375 Brother Yes Yes Fulltime Hired Ahmed Salem Elhossieny Farag Finance & Accounting Financial Administrator Ibrahim Azmy 2016-08-15 00:00:00 2016-11-15 00:00:00 ahmed.salem@paysky.io 1140934077 / 1018994588 Onsite Cairo 28005012101471 Male 1--80 100033801167 CIB No 1026181860 Wife Yes Yes Fulltime Hired Ibrahim Lamlom Mahmoud Ahmed Finance & Accounting General Ledger Supervisor Mohamed Awwad 2021-04-11 00:00:00 2021-07-11 00:00:00 ibrahim.lamloum@paysky.io 1156716741 Onsite Cairo 29101050200256 Male 1991-01-05 00:00:00 34 100047625798 CIB Yes 1272886076 Brother Yes Yes Fulltime Hired Ali Hagag Abdallah Hussien Finance & Accounting Mohamed Awwad 2024-06-02 00:00:00 2024-09-02 00:00:00 ali.hagag@paysky.io 1123085649 Onsite Cairo 29805122301599 Male 1998-05-12 00:00:00 27 100064228948 CIB No 1153386180 Sister No Yes Fulltime Hired Mohamed Awwad Abdelfattah Maghraby Aly Finance & Accounting Accountant Manager Ibrahim Azmy 2024-07-01 00:00:00 2024-10-01 00:00:00 mohamed.awwad@paysky.io 1099669066 Onsite Cairo 28706288800737 Male 1987-06-28 00:00:00 37 100064227615 CIB Yes 1067192783 Wife No Yes Fulltime Resigned Islam Saeed Mohamed Mohamed Raslan Finance & Accounting Financial Operations Supervisor Mohamed Awwad 2024-12-23 00:00:00 2025-03-23 00:00:00 islam.raslan@paysky.io 1014311002 Onsite Cairo 28409141401778 Male 1984-09-14 00:00:00 40 100066968957 CIB 1011833086 Wife No Yes 2025-05-04 00:00:00 Fulltime Hired Nourhan Mousa Abdel Megeed HR HR Director Waleed Sadek 2019-02-01 00:00:00 2019-05-01 00:00:00 nourhan.mousa@paysky.io 1022228244 Hybrid Cairo Nasr City 27912010103461 Female 1979-12-01 00:00:00 45 100039280692 CIB Yes 1118528743 Sister Yes Yes Fulltime Hired Passant Kamal Abdelaziz Shalaby HR HR Operations and Complaince Lead Nourhan Mousa 2021-08-01 00:00:00 2021-11-01 00:00:00 Passant.kamal@paysky.io 1006573331 Hybrid Cairo Maadi 28706120104662 Female 1987-06-12 00:00:00 37 100049115789 CIB Yes 1156060600 Husband Yes Yes Fulltime Hired Elsayed Mahmoud Elsayed Saleh Mahmoud HR HR Business Partner Nourhan Mousa 2023-08-06 00:00:00 2023-11-06 00:00:00 sayed.saleh@paysky.io 1201222483 Hybrid Cairo Zahraa Nasr City 29609060201611 Male 1996-09-06 00:00:00 28 100052374175 CIB Yes 1211682171 Wife Yes Yes Fulltime Hired Yasmine Mahmoud Abdelatty Elsayed Mousa HR Recruitment Specialist Sayed Saleh 2024-10-01 00:00:00 2025-01-01 00:00:00 yasmine.mahmoud@paysky.io 1018135870 Onsite Cairo 30203280102787 Female 2002-03-28 00:00:00 23 100067011297 CIB No 1010510016 Mother No Yes Fulltime Hired Khloud Khaled Mohamed Morsy Marketing Customer Success Customer Success Manager Mohamed Hassan 2021-06-01 00:00:00 2021-09-01 00:00:00 khloud.khaled@paysky.io 1030088511 Hybrid Cairo 29609121400549 Female 1996-09-12 00:00:00 28 100027070757 CIB No 1011108362 Mother Yes Yes Fulltime Hired Eslam Osama Abo bakr Marketing Customer Success Customer Success Specialist Khloud Khaled 2021-06-01 00:00:00 2021-09-01 00:00:00 islam.osama@yalla.online 1029986533 Hybrid Cairo 28803128800477 Male 1988-03-12 00:00:00 37 100049339717 CIB No 1019127600 Sister No Yes Fulltime Hired Donia Wael Mohamed Nabil Abdelfattah Marketing Customer Success Customer Success Specialist Khloud Khaled 2021-06-20 00:00:00 2021-09-20 00:00:00 donia.wael@yalla.online 1154550545 / 1029985415 Hybrid Cairo 29801251600564 Female 1998-01-25 00:00:00 27 100049337242 CIB No 1000012728 Husband Yes Yes Fulltime Hired Hassan Mohamed Mohamed Dergham Shalaby Marketing Growth, Performance and Media Manager Mahmoud Taha 2021-09-01 00:00:00 2021-12-01 00:00:00 Hassan.derghaam@paysky.io 1010643960 Onsite Cairo 29307140400251 Male 1993-07-14 00:00:00 31 100031625135 CIB No 1093917888 Wife Yes Yes Fulltime Hired Abdelrahman Marzouk Mohamed Marzouk Marketing Customer Success Senior Customer Success Specialist Khloud Khaled 2021-12-12 00:00:00 2022-03-12 00:00:00 Abdelrahman.Marzouk @yalla.online 1069497510 Hybrid Cairo 29809192300099 Male 1998-09-19 00:00:00 26 100051738057 CIB No 1008913349 Brother No Yes Fulltime Hired Manal ELsayed AbdelAaty Salem Marketing Customer Success Senior Customer Success Specialist Khloud Khaled 2021-12-12 00:00:00 2022-03-12 00:00:00 manal.elsayed@yalla.online 1155360010 Hybrid Cairo 29712170103903 Female 1997-12-17 00:00:00 27 100051738138 CIB No 1009973583 Sister No Yes Fulltime Hired Abdelrahman ashraf sayed Abdelkader Marketing Customer Success Customer Success Specialist Khloud Khaled 2021-12-15 00:00:00 2022-03-15 00:00:00 abdelrahman.ashraf@yalla.online 1128719522 Hybrid Cairo 30009300118234 Male 2000-09-30 00:00:00 24 100051738106 CIB No 1205556969 Father No Yes Fulltime Hired Mahmoud Mohamed Mahmoud Taha Marketing Chief Marketing Officer - CMO\nCountry Manager - Yalla EGY Waleed Sadek 2022-02-02 00:00:00 2022-05-02 00:00:00 mahmoud.taha@paysky.io 1010405063 Onsite Cairo 28909121200477 Male 1989-09-12 00:00:00 35 100035249766 CIB Yes 1009970033 Brother Yes Yes Fulltime Hired Hisham Moharam Salah Eldin Ibrahim Marketing Marketing Comunication Manager Mahmoud Taha 2022-05-29 00:00:00 2022-08-29 00:00:00 Hisham.Moharam@paysky.io 1212224478 Onsite Cairo Rehab 29106210102894 Male 1991-06-21 00:00:00 33 100054773735 CIB Yes 1210253553 Brother Yes Yes Fulltime Hired Essam AbdelAlim Ibrahim Eloraby Marketing Customer Success Head of Customer Success Mahmoud Taha 2022-10-23 00:00:00 2023-01-23 00:00:00 essam.oraby@paysky.io 1005109513 Onsite Cairo 27607301601291 Male 1976-07-30 00:00:00 48 100048192895 CIB Yes 1001919750 Brother Yes Yes Fulltime Hired Ahmed Samy Bayomi Metwally Marketing Customer Success Customer Success Specialist - UAE Essam Oraby 2022-12-25 00:00:00 2023-03-25 00:00:00 ahmed.bayomi@paysky.io 1064688880 Onsite Cairo 28809260105433 Male 1988-09-26 00:00:00 36 100057143767 CIB 1116765271 Wife Yes Yes Fulltime Hired Omar Abdelaziz Nadir Shehata Mohamed Marketing Customer Success Customer Success Specialist Khloud Khaled 2023-01-15 00:00:00 2023-04-15 00:00:00 omar.abdelaziz@paysky.io 1115630567 Hybrid Cairo 29411210102573 Male 1994-11-21 00:00:00 30 100060193268 CIB No 1221923177 Father Yes Yes Fulltime Hired Alaa Hassan Abdelmohsen Hassan Marketing Senior Performance Specialist Hassan Dorgham 2023-03-19 00:00:00 2023-06-19 00:00:00 Alaa.hassan@paysky.io 1068966860 Onsite Cairo 29508272101526 Female 1995-08-27 00:00:00 29 100043495784 CIB No 1011877887 Husband Yes Yes Fulltime Hired Refaat Ashraf Refaat Batal Technology Data Senior L1 Data Analyst Mohamed Barkat 2025-05-07 00:00:00 2025-08-07 00:00:00 refaat.batal@paysky.io 1553071348 Onsite Cairo 29411260101331 Male 1994-11-26 00:00:00 30 100058209761 CIB Yes 1279016193 Father Yes Yes Fulltime Hired Mohamed elsayed Elewa Salem Shalaby Marketing Customer Success Customer Success Specialist - PAK Essam Oraby 2023-06-11 00:00:00 2023-09-11 00:00:00 mohamed.elewa@paysky.io 1061421111 Onsite Cairo 28511201401194 Male 1985-11-20 00:00:00 39 100049201828 CIB 1027002036 Wife Yes Yes Fulltime Hired Omar Ahmed Tarek Ibrahim Mohamed Marketing BTL Supervisor Hisham Moharam 2023-07-03 00:00:00 2023-10-03 00:00:00 omar.tarek@paysky.io 1140486447 Onsite Cairo 29802140103614 Male 1998-02-14 00:00:00 27 100060193837 CIB Yes 1223484792 Father Yes Yes Fulltime Hired Salwa Amr Beshir Mousa Ahmed Marketing Customer Success Customer Success Specialist Khloud Khaled 2023-08-06 00:00:00 2023-11-06 00:00:00 salwa.amr@yalla.online

In [10]:
prompt = f"""
based on this context {output_retrieval_merged}
answer the following question: {query}
if you don't have information on the answer, say you don't know
"""

In [11]:
model = ChatGroq(groq_api_key=api_key,
                   model = "llama-3.3-70b-versatile",
                   temperature=0)
response_text= model.invoke(prompt)

In [12]:
display(Markdown(response_text.content))

I don't know. The provided context does not contain any review information, only employee data.

In [13]:
#Functions for Excel
def prepare_excel(file_path):
    loader = UnstructuredExcelLoader(file_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    chunks = text_splitter.split_documents(data)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db_faiss = FAISS.from_documents(chunks, embeddings)

    return db_faiss


In [14]:
def rag(query, db_faiss, k= 3):
    output_retrieval = db_faiss.similarity_search(query, k= k)

    output_retrieval_merged = "\n\n".join([doc.page_content for doc in output_retrieval])

    prompt = f"""
    based on this context {output_retrieval_merged}
    answer the following question: {query}
    if you don't have information on the answer, say you don't know
    """

    model = ChatGroq(groq_api_key=api_key,
                   model = "llama-3.3-70b-versatile",
                   temperature=0)
    response_text= model.invoke(prompt)

    return display(Markdown(response_text.content))


In [21]:
db_excel = prepare_excel("Sample of Employee Database.xlsx")

In [26]:
rag("Who has a car and is living in madinaty", db_excel)

Based on the provided context, the person who has a car and is living in Madinaty is Karim Salah Abdelwahab Ahmed Khattab.